# 📈 Datathon Template — TIME SERIES FORECASTING

এক বা একাধিক series (যেমন store × item এর daily sales) এর **ভবিষ্যৎ মান predict** করা।

| # | Step | কি করব |
|---|------|--------|
| 0 | Colab setup & Config | date column, target, series ID, frequency, horizon |
| 1 | Load data | |
| 2 | Cleaning | duplicate (same date+series), **missing dates**, missing value, negative, outlier |
| 3 | EDA | trend, seasonality (day/week/month/year), decomposition (STL), ACF/PACF, stationarity (ADF), promo/exog effect |
| 4 | Validation strategy | **time-based** holdout + backtesting (কখনো random split না!) |
| 5 | Baselines | naive, seasonal naive, moving average, seasonal average |
| 6 | Statistical models | Exponential Smoothing (Holt-Winters), SARIMA, Prophet |
| 7 | Feature engineering | calendar, holiday, **lag**, rolling, EWM, exogenous |
| 8 | ML Model zoo | Ridge, RF, ExtraTrees, HistGB, LightGBM, XGBoost, CatBoost (একটা global model সব series এর জন্য) |
| 9 | Backtesting + Tuning | multiple fold, Optuna |
| 10 | Evaluation | forecast plot, error by horizon / series, residual |
| 11 | Interpretation | feature importance, SHAP |
| 12 | Ensemble + Final forecast + Submission | |

> **Golden rule:** model কে কখনো ভবিষ্যতের তথ্য দেখতে দেওয়া যাবে না। Validation সবসময় **train এর পরের সময়ের** data।

## ☁️ Google Colab Setup (সবার আগে এটা চালাও)

**GPU চালু করা:** `Runtime → Change runtime type → T4 GPU → Save` (image / transformer এর জন্য জরুরি)

**Data আনার ৪টা উপায় — নিচের `DATA_SOURCE` বদলাও:**
| `DATA_SOURCE` | কিভাবে |
|---|---|
| `"demo"` | কিছু লাগবে না — notebook নিজেই practice data বানাবে |
| `"upload"` | Cell চালালে **Choose Files** button আসবে → `train.csv`, `test.csv`, `sample_submission.csv` (বা একটা `.zip`) select করো |
| `"drive"` | আগে Google Drive এ data রাখো (যেমন `MyDrive/datathon/timeseries/`) → `DRIVE_DATA_DIR` ঠিক করো → permission দাও |
| `"kaggle"` | Kaggle → Settings → **Create New Token** → `kaggle.json` download → cell চালালে upload করো; competition page এ আগে **rules accept** করতে হবে |

সব data শেষে `data/timeseries` folder এ আসবে, `.zip` থাকলে auto unzip হবে — তাই নিচের CONFIG এ path বদলাতে হবে না
(শুধু file / column এর নাম মিলিয়ে নিও)।

> ⚠️ Colab বন্ধ হলে `/content` এর সব file মুছে যায় — submission আর model notebook এর শেষ cell দিয়ে download / Drive এ save করো।

In [ ]:
import sys, os, shutil, zipfile, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# ---------- 1) যে library Colab এ নেই সেগুলো install ----------
NEED = {"seaborn": "seaborn", "statsmodels": "statsmodels", "lightgbm": "lightgbm", "xgboost": "xgboost", "catboost": "catboost", "optuna": "optuna", "shap": "shap", "holidays": "holidays"}          # {import_name: pip_name}
missing = [pip for mod, pip in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

# ---------- 2) Data ----------
DATA_SOURCE = "demo"                                  # "demo" | "upload" | "drive" | "kaggle"
DRIVE_DATA_DIR = "/content/drive/MyDrive/datathon/timeseries"
KAGGLE_COMPETITION = ""                               # e.g. "titanic"  (URL: kaggle.com/competitions/<এই নাম>)
KAGGLE_DATASET = ""                                   # competition না হয়ে dataset হলে: "owner/dataset-name"
LOCAL_DATA_DIR = Path("data/timeseries")

def unzip_all(folder):
    while zips := list(Path(folder).rglob("*.zip")):                       # zip এর ভেতরে zip থাকলেও (Kaggle এ common)
        for z in zips:
            print("Unzipping", z.name)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(z.parent)
            z.unlink()

if IN_COLAB and DATA_SOURCE != "demo":
    from google.colab import files, drive
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    if DATA_SOURCE == "upload":
        for name in files.upload():
            shutil.move(name, LOCAL_DATA_DIR / name)
    elif DATA_SOURCE == "drive":
        drive.mount("/content/drive")
        shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR, dirs_exist_ok=True)   # local copy → Drive থেকে পড়ার চেয়ে অনেক দ্রুত
    elif DATA_SOURCE == "kaggle":
        # নতুন token (KGAT_...) বা পুরনো kaggle.json — যেটা আছে সেটা দিয়েই চলবে
        import getpass
        if not os.environ.get("KAGGLE_API_TOKEN") and not Path("~/.kaggle/kaggle.json").expanduser().exists():
            tok = getpass.getpass("Kaggle API token paste করো (লুকানো থাকবে, KGAT_... দিয়ে শুরু) অথবা খালি রেখে Enter দিয়ে kaggle.json upload করো: ").strip()
            if tok:
                os.environ["KAGGLE_API_TOKEN"] = tok
            else:
                files.upload()
                Path("~/.kaggle").expanduser().mkdir(exist_ok=True)
                shutil.move("kaggle.json", Path("~/.kaggle/kaggle.json").expanduser())
                os.chmod(Path("~/.kaggle/kaggle.json").expanduser(), 0o600)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=False)
        if KAGGLE_COMPETITION:
            subprocess.run(["kaggle", "competitions", "download", "-c", KAGGLE_COMPETITION, "-p", str(LOCAL_DATA_DIR)], check=True)
        else:
            subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(LOCAL_DATA_DIR)], check=True)
    unzip_all(LOCAL_DATA_DIR)

if LOCAL_DATA_DIR.exists():
    print(f"\n📁 {LOCAL_DATA_DIR} এর ভেতরে:")
    for p in sorted(LOCAL_DATA_DIR.iterdir())[:30]:
        print("  ", p.name, "/" if p.is_dir() else f"({p.stat().st_size / 1e6:.2f} MB)")
print("IN_COLAB =", IN_COLAB, "| DATA_SOURCE =", DATA_SOURCE)

## 0. Setup & Config

In [ ]:
import os, random, time, warnings, importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.base import clone

def try_import(name):
    try:
        return importlib.import_module(name)
    except ImportError:
        print(f"⚠️  '{name}' নেই → pip install {name} (এই অংশ skip হবে)")
        return None

sm = try_import("statsmodels")
lgb = try_import("lightgbm")
xgb = try_import("xgboost")
cb = try_import("catboost")
optuna = try_import("optuna")
shap = try_import("shap")
holidays_lib = try_import("holidays")
prophet = try_import("prophet")

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

In [ ]:
# ======================= CONFIG =======================
DATA_DIR = Path("data/timeseries")
TRAIN_FILE = DATA_DIR / "train.csv"
TEST_FILE = DATA_DIR / "test.csv"                  # ভবিষ্যতের date গুলো (target ছাড়া)
SUBMISSION_FILE = DATA_DIR / "sample_submission.csv"
OUTPUT_DIR = Path("outputs/timeseries")

DATE_COL = "date"
TARGET = "sales"
ID_COLS = ["store", "item"]       # প্রতিটা series চেনার column; একটাই series হলে []
ROW_ID_COL = "id"                 # submission এর id column (test এ থাকে); না থাকলে None
EXOG_COLS = None                  # None → auto: train আর test দুটোতেই আছে এমন বাকি column (ভবিষ্যতে জানা, যেমন promo, price)

FREQ = "D"                        # "D" daily, "W" weekly, "MS" monthly, "h" hourly
SEASON = 7                        # মূল seasonal period: daily→7, weekly→52, monthly→12, hourly→24
HORIZON = None                    # কত step সামনে predict; None → test থেকে auto
COUNTRY = "BD"                    # holiday feature (holidays library); None → skip

METRIC = "rmse"                   # "rmse" | "mae" | "mape" | "smape" | "rmsle" | "wape"
LOG_TARGET = True                 # target ≥ 0 আর skewed (sales/count) হলে log1p এ train করো
NONNEGATIVE = True                # prediction negative হতে পারবে না
FILL_MISSING_DATES = "zero"       # missing date এর target: "zero" (sales নেই) | "interpolate" | "ffill" | None
CLIP_OUTLIERS = True
N_BACKTEST = 3                    # কয়টা time fold এ backtest
MAX_STAT_SERIES = 50              # এর বেশি series হলে ETS (per-series model) skip — slow
N_OPTUNA_TRIALS = 20
SEED = 42
# ======================================================

random.seed(SEED); np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### (Practice) Demo dataset generator
৩ store × ৪ item = ১২টা daily sales series (৩ বছর), সাথে **trend, weekly + yearly seasonality, promo, price**।
ইচ্ছা করে রাখা আছে: missing date, NaN, outlier, duplicate row, আর এমন একটা column (`customers`) যেটা ভবিষ্যতে জানা যায় না।

In [ ]:
def make_demo_ts(seed=SEED, horizon=28):
    rng = np.random.default_rng(seed)
    tr_dates = pd.date_range("2021-01-01", "2023-12-31", freq="D")
    all_dates = tr_dates.append(pd.date_range(tr_dates[-1] + pd.Timedelta(days=1), periods=horizon, freq="D"))
    t = np.arange(len(all_dates))
    dow_eff = np.array([0.9, 0.85, 0.9, 1.0, 1.15, 1.35, 1.2])[all_dates.dayofweek]
    year_eff = 1 + 0.25 * np.sin(2 * np.pi * (all_dates.dayofyear - 80) / 365.25)
    frames = []
    for s in range(3):
        for i in range(4):
            base, trend = rng.uniform(15, 70), rng.uniform(0.005, 0.03)
            promo = (rng.random(len(t)) < 0.1).astype(int)
            price = np.round(rng.uniform(5, 50) * (1 - 0.15 * promo), 2)
            mu = (base + trend * t) * dow_eff * year_eff * (1 + 0.4 * promo) * (1 + 0.1 * s)
            sales = rng.poisson(mu).astype(float)
            frames.append(pd.DataFrame({"date": all_dates, "store": f"S{s + 1}", "item": f"I{i + 1}", "promo": promo,
                                        "price": price, "customers": rng.poisson(mu * 0.6), "sales": sales}))
    df = pd.concat(frames, ignore_index=True)
    train = df[df.date <= tr_dates[-1]].copy()
    test = df[df.date > tr_dates[-1]].drop(columns=["sales", "customers"]).copy()

    # ---- make it messy ----
    train = train.drop(train.sample(frac=0.01, random_state=seed).index)                 # missing dates
    train.loc[train.sample(frac=0.01, random_state=seed + 1).index, "sales"] = np.nan     # missing values
    out = train.sample(frac=0.002, random_state=seed + 2).index
    train.loc[out, "sales"] = train.loc[out, "sales"] * 6                                 # outliers
    train = pd.concat([train, train.sample(20, random_state=seed)])                        # duplicates
    train["date"] = train["date"].dt.strftime("%Y-%m-%d"); test["date"] = test["date"].dt.strftime("%Y-%m-%d")
    test.insert(0, "id", np.arange(len(test)))
    return train.sample(frac=1, random_state=seed), test, pd.DataFrame({"id": test["id"], "sales": 0})

if not TRAIN_FILE.exists():
    print("Data পাওয়া যায়নি → DEMO time-series dataset বানানো হচ্ছে")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    _tr, _te, _sub = make_demo_ts()
    _tr.to_csv(TRAIN_FILE, index=False); _te.to_csv(TEST_FILE, index=False); _sub.to_csv(SUBMISSION_FILE, index=False)

## 1. Load data

In [ ]:
train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE) if TEST_FILE.exists() else None
sample_sub = pd.read_csv(SUBMISSION_FILE) if SUBMISSION_FILE.exists() else None

for df in [train, test]:
    if df is not None:
        df[DATE_COL] = pd.to_datetime(df[DATE_COL])
        df["_sid"] = df[ID_COLS].astype(str).agg("_".join, axis=1) if ID_COLS else "all"

train = train.sort_values(["_sid", DATE_COL]).reset_index(drop=True)
TRAIN_END = train[DATE_COL].max()
print(f"train: {train.shape} | {train[DATE_COL].min().date()} → {TRAIN_END.date()} | series: {train._sid.nunique()}")
if test is not None:
    print(f"test : {test.shape} | {test[DATE_COL].min().date()} → {test[DATE_COL].max().date()}")
    H = len(pd.date_range(TRAIN_END, test[DATE_COL].max(), freq=FREQ)) - 1
else:
    H = HORIZON or 28
H = HORIZON or H
print(f"Forecast HORIZON = {H} steps ({FREQ})")

if EXOG_COLS is None:
    EXOG_COLS = [c for c in (test.columns if test is not None else []) if c in train.columns
                 and c not in {DATE_COL, TARGET, ROW_ID_COL, "_sid", *ID_COLS}]
future_unknown = [c for c in train.columns if c not in EXOG_COLS + [DATE_COL, TARGET, ROW_ID_COL, "_sid", *ID_COLS]]
print("Exogenous (ভবিষ্যতে জানা) :", EXOG_COLS)
print("Train-only (ভবিষ্যতে জানা না → feature হিসেবে নেওয়া যাবে না):", future_unknown)
display(train.head())

## 2. Cleaning
### 2.1 Duplicate (same series + same date)

In [ ]:
dup = train.duplicated(["_sid", DATE_COL], keep=False)
print(f"Duplicate (series, date) rows: {dup.sum()}")
agg = {c: "first" for c in train.columns if c not in ["_sid", DATE_COL]}
agg[TARGET] = "mean"                       # conflicting হলে গড় (sum দরকার হলে "sum")
train = train.groupby(["_sid", DATE_COL], as_index=False).agg(agg)

### 2.2 Missing dates → প্রতিটা series এ পুরো date range
Time series এ **date gap** থাকলে lag feature ভুল হয় (lag_7 আসলে ৭ দিন আগের হবে না)। তাই আগে সব date পূরণ করি।

In [ ]:
def complete_dates(df):
    out = []
    for sid, g in df.groupby("_sid"):
        idx = pd.date_range(g[DATE_COL].min(), TRAIN_END, freq=FREQ)
        g = g.set_index(DATE_COL).reindex(idx).rename_axis(DATE_COL)
        g["_filled_date"] = g["_sid"].isna().astype(int)
        g["_sid"] = sid
        for c in ID_COLS + EXOG_COLS:
            g[c] = g[c].ffill().bfill()
        out.append(g.reset_index())
    return pd.concat(out, ignore_index=True)

n_before = len(train)
train = complete_dates(train)
print(f"Missing dates added: {len(train) - n_before}")
print(f"Missing target values (NaN): {train[TARGET].isna().sum()}  (filled-date rows: {train._filled_date.sum()})")

if FILL_MISSING_DATES == "zero":
    train.loc[train._filled_date == 1, TARGET] = 0
# বাকি NaN (data তে আসলেই missing) → series অনুযায়ী time interpolation
if FILL_MISSING_DATES in ("zero", "interpolate"):
    train[TARGET] = train.groupby("_sid")[TARGET].transform(lambda s: s.interpolate(limit_direction="both"))
elif FILL_MISSING_DATES == "ffill":
    train[TARGET] = train.groupby("_sid")[TARGET].transform(lambda s: s.ffill().bfill())
if NONNEGATIVE:
    print("Negative targets:", (train[TARGET] < 0).sum()); train[TARGET] = train[TARGET].clip(lower=0)
print("NaN left:", train[TARGET].isna().sum())

### 2.3 Outliers — rolling median + MAD (robust z-score)
সাধারণ IQR time series এ কাজ করে না (trend/seasonality থাকে), তাই **local** median থেকে কতটা দূরে সেটা দেখি।

In [ ]:
def robust_z(s, w=SEASON * 4 + 1):
    med = s.rolling(w, center=True, min_periods=SEASON).median()
    mad = (s - med).abs().rolling(w, center=True, min_periods=SEASON).median()
    return (s - med) / (1.4826 * mad + 1e-9), med

z, med = [], []
for _, g in train.groupby("_sid")[TARGET]:
    zz, mm = robust_z(g); z.append(zz); med.append(mm)
train["_z"], train["_med"] = pd.concat(z), pd.concat(med)
out_mask = train["_z"].abs() > 6
print(f"Outliers found: {out_mask.sum()} ({out_mask.mean():.2%})")

ex_sid = train.loc[out_mask, "_sid"].iloc[0] if out_mask.any() else train._sid.iloc[0]
g = train[train._sid == ex_sid]
plt.figure(figsize=(15, 3.5))
plt.plot(g[DATE_COL], g[TARGET], lw=.7, label=TARGET); plt.plot(g[DATE_COL], g["_med"], lw=1, label="rolling median")
plt.scatter(g.loc[out_mask[g.index], DATE_COL], g.loc[out_mask[g.index], TARGET], c="r", s=25, label="outlier", zorder=3)
plt.title(f"Outlier detection — {ex_sid}"); plt.legend(); plt.show()

if CLIP_OUTLIERS:
    train.loc[out_mask, TARGET] = train.loc[out_mask, "_med"]     # outlier → local median
train = train.drop(columns=["_z", "_med"])

## 3. EDA
### 3.1 Overall trend + sample series

In [ ]:
total = train.groupby(DATE_COL)[TARGET].sum()
fig, axes = plt.subplots(2, 1, figsize=(16, 7))
axes[0].plot(total.index, total.values, lw=.6, alpha=.6, label="total")
axes[0].plot(total.index, total.rolling(SEASON).mean(), lw=1.5, label=f"rolling {SEASON}")
axes[0].plot(total.index, total.rolling(SEASON * 13).mean(), lw=2, label=f"rolling {SEASON * 13}")
if test is not None:
    axes[0].axvspan(test[DATE_COL].min(), test[DATE_COL].max(), color="orange", alpha=.3, label="test period")
axes[0].legend(); axes[0].set_title("Total target over time")
top_sids = train.groupby("_sid")[TARGET].sum().sort_values(ascending=False).index[:6]
for sid in top_sids:
    s = train[train._sid == sid].set_index(DATE_COL)[TARGET]
    axes[1].plot(s.index, s.rolling(SEASON).mean(), lw=1, label=sid)
axes[1].legend(ncol=6, fontsize=8); axes[1].set_title(f"Top series (rolling {SEASON} mean)")
plt.tight_layout(); plt.show()

In [ ]:
# Series-level summary — কোন series বড়, কোনটায় অনেক zero (intermittent demand)?
series_stats = train.groupby("_sid")[TARGET].agg(start=lambda s: train.loc[s.index, DATE_COL].min().date(), length="size",
                                                  mean="mean", std="std", zero_pct=lambda s: (s == 0).mean() * 100)
series_stats["cv"] = series_stats["std"] / (series_stats["mean"] + 1e-9)
display(series_stats.sort_values("mean", ascending=False).head(15))

### 3.2 Seasonality — day of week, month, year

In [ ]:
tmp = train[[DATE_COL, TARGET]].copy()
tmp["dow"], tmp["month"], tmp["year"] = tmp[DATE_COL].dt.dayofweek, tmp[DATE_COL].dt.month, tmp[DATE_COL].dt.year
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.barplot(data=tmp, x="dow", y=TARGET, ax=axes[0], errorbar=None); axes[0].set_title("Mean by day of week (0=Mon)")
sns.barplot(data=tmp, x="month", y=TARGET, ax=axes[1], errorbar=None); axes[1].set_title("Mean by month")
sns.barplot(data=tmp, x="year", y=TARGET, ax=axes[2], errorbar=None); axes[2].set_title("Mean by year (trend)")
plt.tight_layout(); plt.show()

pv = tmp.pivot_table(index="year", columns="month", values=TARGET, aggfunc="mean")
plt.figure(figsize=(12, 2.5 + .3 * len(pv)))
sns.heatmap(pv, annot=True, fmt=".0f", cmap="YlGnBu"); plt.title("Year × Month heatmap"); plt.show()

### 3.3 Decomposition (STL) + ACF/PACF + Stationarity (ADF)
- **STL:** series = trend + seasonal + residual
- **ACF:** কোন lag এর সাথে correlation বেশি → কোন lag feature কাজে লাগবে (daily data তে 7, 14, 28 ... spike মানে weekly pattern)
- **ADF test:** p < 0.05 → stationary। না হলে ARIMA তে differencing (`d=1`) লাগবে

In [ ]:
if sm is not None:
    from statsmodels.tsa.seasonal import STL
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.tsa.stattools import adfuller

    stl = STL(total, period=SEASON, robust=True).fit()
    fig = stl.plot(); fig.set_size_inches(15, 8); plt.show()
    print(f"Trend strength    = {max(0, 1 - stl.resid.var() / (stl.trend + stl.resid).var()):.2f}")
    print(f"Seasonal strength = {max(0, 1 - stl.resid.var() / (stl.seasonal + stl.resid).var()):.2f}   (০→নেই, ১→খুব শক্তিশালী)")

    fig, axes = plt.subplots(1, 2, figsize=(16, 3.8))
    plot_acf(total, lags=min(SEASON * 8, len(total) // 3), ax=axes[0])
    plot_pacf(total, lags=min(SEASON * 4, len(total) // 3), ax=axes[1], method="ywm")
    plt.tight_layout(); plt.show()

    for name, s in [("original", total), ("1st difference", total.diff().dropna()), (f"seasonal diff ({SEASON})", total.diff(SEASON).dropna())]:
        p = adfuller(s.values, autolag="AIC")[1]
        print(f"ADF {name:22s} p-value = {p:.4f} → {'stationary ✅' if p < .05 else 'non-stationary ⚠️'}")

### 3.4 Target distribution + exogenous effect

In [ ]:
fig, axes = plt.subplots(1, 2 + len(EXOG_COLS[:3]), figsize=(5.5 * (2 + len(EXOG_COLS[:3])), 3.8))
sns.histplot(train[TARGET], bins=60, ax=axes[0]); axes[0].set_title(f"{TARGET} (skew={train[TARGET].skew():.2f})")
sns.histplot(np.log1p(train[TARGET].clip(lower=0)), bins=60, ax=axes[1], color="g"); axes[1].set_title("log1p(target)")
for ax, c in zip(axes[2:], EXOG_COLS[:3]):
    if train[c].nunique() <= 10:
        sns.boxplot(data=train, x=c, y=TARGET, ax=ax, showfliers=False)
    else:
        smp = train.sample(min(3000, len(train)), random_state=SEED)
        sns.scatterplot(data=smp, x=c, y=TARGET, s=6, alpha=.4, ax=ax)
    ax.set_title(f"{TARGET} vs {c}")
plt.tight_layout(); plt.show()

## 4. Validation Strategy — time-based
- **Holdout:** train এর শেষ `H` step = validation (test যেমন train এর ঠিক পরে)
- **Backtesting:** একাধিক cutoff এ একই কাজ → score stable কিনা দেখা
- ⚠️ `KFold(shuffle=True)` / random split **নিষেধ** — ভবিষ্যৎ দিয়ে অতীত শেখা হয়ে যায় (leakage)

In [ ]:
STEP = pd.tseries.frequencies.to_offset(FREQ)
VALID_START = TRAIN_END - (H - 1) * STEP
FOLD_STARTS = [VALID_START - k * H * STEP for k in range(N_BACKTEST)][::-1]    # backtest cutoffs (পুরনো → নতুন)

fig, ax = plt.subplots(figsize=(14, 1 + .45 * N_BACKTEST))
for i, fs in enumerate(FOLD_STARTS):
    ax.plot([train[DATE_COL].min(), fs], [i, i], lw=14, color="steelblue", solid_capstyle="butt")
    ax.plot([fs, fs + (H - 1) * STEP], [i, i], lw=14, color="orange", solid_capstyle="butt")
ax.set_yticks(range(N_BACKTEST)); ax.set_yticklabels([f"fold {i}" for i in range(N_BACKTEST)])
ax.set_title("Backtest folds (blue = train, orange = validation)"); plt.show()
print("Validation (holdout):", VALID_START.date(), "→", TRAIN_END.date())

## 5. Metrics + Baselines
**Baseline ছাড়া কোনো model ভালো কিনা বোঝা যায় না।** Time series এ seasonal naive প্রায়ই অবাক করার মতো শক্তিশালী।

In [ ]:
def _clip(p): return np.clip(p, 0, None) if NONNEGATIVE else p
METRICS = {
    "rmse": lambda y, p: np.sqrt(np.mean((y - p) ** 2)),
    "mae": lambda y, p: np.mean(np.abs(y - p)),
    "mape": lambda y, p: 100 * np.mean(np.abs(y - p)[y != 0] / np.abs(y[y != 0])),
    "smape": lambda y, p: 100 * np.mean(2 * np.abs(p - y) / (np.abs(y) + np.abs(p) + 1e-9)),
    "rmsle": lambda y, p: np.sqrt(np.mean((np.log1p(np.clip(p, 0, None)) - np.log1p(np.clip(y, 0, None))) ** 2)),
    "wape": lambda y, p: 100 * np.sum(np.abs(y - p)) / (np.sum(np.abs(y)) + 1e-9),
}
def score(y, p, metric=METRIC):
    return METRICS[metric](np.asarray(y, float), np.asarray(p, float))      # lower = better

def per_series_forecast(fn, hist_df, target_df):
    """fn(history_array, h) → h-step forecast। target_df এর প্রতিটা row এর date অনুযায়ী সঠিক step বসায়।"""
    preds = pd.Series(np.nan, index=target_df.index)
    hist_groups = {sid: g.sort_values(DATE_COL) for sid, g in hist_df.groupby("_sid")}
    for sid, g in target_df.groupby("_sid"):
        hg = hist_groups.get(sid)
        if hg is None or hg[TARGET].notna().sum() == 0:
            preds[g.index] = 0; continue
        last = hg[DATE_COL].max()
        h = len(pd.date_range(last, g[DATE_COL].max(), freq=FREQ)) - 1
        future = pd.date_range(last, periods=h + 1, freq=FREQ)[1:]
        fc = np.asarray(fn(hg[TARGET].dropna().values.astype(float), h), float)
        preds[g.index] = fc[pd.Index(future).get_indexer(g[DATE_COL])]
    return _clip(preds.values)

def seasonal_avg(hist, h, k=4):
    k = max(1, min(k, len(hist) // SEASON))
    if len(hist) < SEASON: return np.repeat(hist.mean(), h)
    return np.resize(hist[-k * SEASON:].reshape(k, SEASON).mean(0), h)

BASELINES = {
    "Naive (last value)": lambda hist, h: np.repeat(hist[-1], h),
    "Seasonal naive": lambda hist, h: np.resize(hist[-SEASON:], h),
    "Moving average": lambda hist, h: np.repeat(hist[-4 * SEASON:].mean(), h),
    "Seasonal average (4)": seasonal_avg,
}

## 6. Statistical models
### 6.1 Exponential Smoothing / Holt-Winters (প্রতিটা series এ আলাদা model)

In [ ]:
STAT_MODELS = {}
if sm is not None and train._sid.nunique() <= MAX_STAT_SERIES:
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

    def ets_fn(hist, h):
        y = np.log1p(np.clip(hist, 0, None)) if LOG_TARGET else hist
        try:
            seasonal = "add" if len(y) >= 2 * SEASON else None
            fit = ExponentialSmoothing(y, trend="add", damped_trend=True, seasonal=seasonal,
                                       seasonal_periods=SEASON if seasonal else None).fit(optimized=True)
            fc = fit.forecast(h)
        except Exception:
            fc = np.repeat(y[-SEASON:].mean(), h)
        return np.expm1(fc) if LOG_TARGET else fc

    STAT_MODELS["ETS (Holt-Winters)"] = ets_fn
else:
    print("ETS skipped (statsmodels নেই বা series অনেক বেশি)")

### 6.2 SARIMA + Prophet (aggregated total series এর উপর — শেখার জন্য)
`SARIMAX(order=(p,d,q), seasonal_order=(P,D,Q,s))` — ACF/PACF দেখে p, q; ADF দেখে d।
অনেক series হলে per-series SARIMA খুব slow → তখন ML model (Section 8) ভালো।

In [ ]:
tot_tr = total[total.index < VALID_START]; tot_va = total[total.index >= VALID_START]
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(tot_tr.index[-SEASON * 12:], tot_tr.values[-SEASON * 12:], label="history", color="gray")
ax.plot(tot_va.index, tot_va.values, label="actual", color="k", lw=2)
if sm is not None:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    t0 = time.time()
    sar = SARIMAX(np.log1p(tot_tr.values) if LOG_TARGET else tot_tr.values, order=(1, 1, 1),
                  seasonal_order=(1, 1, 1, SEASON)).fit(disp=False)
    sar_fc = sar.forecast(len(tot_va)); sar_fc = np.expm1(sar_fc) if LOG_TARGET else sar_fc
    ax.plot(tot_va.index, sar_fc, label=f"SARIMA ({METRIC}={score(tot_va.values, sar_fc):.1f})")
    print(f"SARIMA fit {time.time() - t0:.1f}s | AIC = {sar.aic:.1f}")
if prophet is not None:
    try:
        from prophet import Prophet
        pm = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
        pm.fit(pd.DataFrame({"ds": tot_tr.index, "y": tot_tr.values}))
        pr_fc = pm.predict(pd.DataFrame({"ds": tot_va.index}))["yhat"].values
        ax.plot(tot_va.index, pr_fc, label=f"Prophet ({METRIC}={score(tot_va.values, pr_fc):.1f})")
    except Exception as e:
        print("Prophet skipped:", type(e).__name__)
sn = np.resize(tot_tr.values[-SEASON:], len(tot_va))
ax.plot(tot_va.index, sn, "--", label=f"Seasonal naive ({METRIC}={score(tot_va.values, sn):.1f})")
ax.legend(); ax.set_title("Aggregated series — statistical models on validation"); plt.show()

## 7. Feature Engineering (ML model এর জন্য)
**Direct strategy:** সব lag ≥ `H` রাখি। তাহলে test এর শেষ দিন predict করতেও শুধু train এর জানা data লাগে —
কোনো leakage নেই, recursive prediction ও লাগে না।

| Feature | উদাহরণ |
|---|---|
| Calendar | dayofweek, month, weekofyear, dayofyear, is_weekend, month start/end, sin/cos |
| Holiday | `holidays` library (BD, US ...) |
| Lag | `lag_H ... lag_{H+6}`, `lag_364` (গত বছর একই দিন) |
| Rolling (shift H এর পরে) | mean, std, min, max — window 7/14/28/91 |
| EWM | exponentially weighted mean |
| Exogenous | promo, price (test এ দেওয়া থাকে) |
| Series ID | store, item (category code) |

In [ ]:
LAGS = sorted(set(range(H, H + SEASON)) | {l for l in [SEASON * k for k in range(1, 9)] + [364, 365] if l >= H})
LAGS = [l for l in LAGS if l < train.groupby("_sid").size().max() * 0.8]
ROLL_WINDOWS = [SEASON, SEASON * 2, SEASON * 4, SEASON * 13]
print("Lags:", LAGS, "| Rolling windows:", ROLL_WINDOWS)

full = pd.concat([train.assign(_is_test=0), (test.assign(_is_test=1) if test is not None else None)], ignore_index=True)
full = full.sort_values(["_sid", DATE_COL]).reset_index(drop=True)

def add_features(df):
    d = df[DATE_COL]
    df["year"], df["month"], df["day"] = d.dt.year, d.dt.month, d.dt.day
    df["dayofweek"], df["dayofyear"], df["quarter"] = d.dt.dayofweek, d.dt.dayofyear, d.dt.quarter
    df["weekofyear"] = d.dt.isocalendar().week.astype(int).values
    df["is_weekend"] = (d.dt.dayofweek >= 5).astype(int)
    df["is_month_start"], df["is_month_end"] = d.dt.is_month_start.astype(int), d.dt.is_month_end.astype(int)
    if FREQ.lower().startswith("h"):
        df["hour"] = d.dt.hour
    for name, period, col in [("dow", 7, "dayofweek"), ("month", 12, "month"), ("doy", 365.25, "dayofyear")]:
        df[f"{name}_sin"], df[f"{name}_cos"] = np.sin(2 * np.pi * df[col] / period), np.cos(2 * np.pi * df[col] / period)
    df["time_idx"] = (d - d.min()).dt.days                                    # trend
    if holidays_lib is not None and COUNTRY:
        hd = holidays_lib.country_holidays(COUNTRY, years=range(d.dt.year.min(), d.dt.year.max() + 1))
        hdates = pd.to_datetime(list(hd.keys()))
        df["is_holiday"] = d.dt.normalize().isin(hdates).astype(int)

    df["_y"] = np.log1p(df[TARGET].clip(lower=0)) if LOG_TARGET else df[TARGET]
    g = df.groupby("_sid")["_y"]
    for l in LAGS:
        df[f"lag_{l}"] = g.shift(l)
    shifted = g.shift(H)
    gs = shifted.groupby(df["_sid"])
    for w in ROLL_WINDOWS:
        df[f"roll_mean_{w}"] = gs.transform(lambda s: s.rolling(w, min_periods=1).mean())
        df[f"roll_std_{w}"] = gs.transform(lambda s: s.rolling(w, min_periods=2).std())
    df[f"roll_max_{ROLL_WINDOWS[2]}"] = gs.transform(lambda s: s.rolling(ROLL_WINDOWS[2], min_periods=1).max())
    df[f"roll_min_{ROLL_WINDOWS[2]}"] = gs.transform(lambda s: s.rolling(ROLL_WINDOWS[2], min_periods=1).min())
    for a in [0.05, 0.2]:
        df[f"ewm_{a}"] = gs.transform(lambda s: s.ewm(alpha=a).mean())
    seas = [f"lag_{l}" for l in LAGS if l % SEASON == 0][:4]
    if seas:
        df["seasonal_lag_mean"] = df[seas].mean(axis=1)                     # একই weekday এর গত কয়েক সপ্তাহের গড়
    df["lag_ratio"] = df[f"lag_{H}"] / (df[f"roll_mean_{ROLL_WINDOWS[2]}"] + 1e-6)

    df["_sid_code"] = df["_sid"].astype("category").cat.codes                # series identity
    for c in ID_COLS + EXOG_COLS:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype("category").cat.codes
    return df

t0 = time.time()
feat = add_features(full)
DROP = {DATE_COL, TARGET, "_y", "_sid", "_is_test", "_filled_date", ROW_ID_COL, *future_unknown}
FEATS = [c for c in feat.columns if c not in DROP and pd.api.types.is_numeric_dtype(feat[c])]
print(f"{len(FEATS)} features in {time.time() - t0:.1f}s:\n", FEATS)

In [ ]:
# Lag correlation check — কোন lag target এর সাথে সবচেয়ে related?
lc = feat.loc[feat._is_test == 0, [f"lag_{l}" for l in LAGS] + ["_y"]].corr()["_y"].drop("_y").sort_values()
lc.plot.barh(figsize=(7, .3 * len(lc) + 1), title="Correlation of lag features with target"); plt.show()

## 8. Model Zoo — global ML model (সব series একসাথে) + baselines + ETS
সব model একই **unified runner** দিয়ে চলে: `run_model(name, cutoff, target_rows)` → cutoff এর আগের data তে train,
target rows predict। Validation, backtest আর final test — তিনটাতেই একই function।

In [ ]:
def ml_models():
    m = {
        "Ridge": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1.0)),
        "RandomForest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=200, min_samples_leaf=3, max_features=0.5, n_jobs=-1, random_state=SEED)),
        "ExtraTrees": make_pipeline(SimpleImputer(strategy="median"), ExtraTreesRegressor(n_estimators=200, min_samples_leaf=3, max_features=0.5, n_jobs=-1, random_state=SEED)),
        "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, random_state=SEED),
    }
    if is_train_row.sum() > 300_000:                    # বড় data তে RF/ET খুব slow → বাদ (Colab এ ঘণ্টা লাগবে)
        m.pop("RandomForest"); m.pop("ExtraTrees")
    if lgb: m["LightGBM"] = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=63, subsample=.8, subsample_freq=1, colsample_bytree=.8, random_state=SEED, verbose=-1)
    if xgb: m["XGBoost"] = xgb.XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=7, subsample=.8, colsample_bytree=.8, random_state=SEED, n_jobs=-1, verbosity=0)
    if cb: m["CatBoost"] = cb.CatBoostRegressor(iterations=800, learning_rate=0.05, depth=7, random_seed=SEED, verbose=0)
    return m

is_train_row = (feat._is_test == 0) & feat[f"lag_{H}"].notna()
ML = ml_models()

def run_model(name, cutoff, target_idx):
    """cutoff এর আগের সব data দিয়ে train → feat.loc[target_idx] predict (original scale)।"""
    tgt = feat.loc[target_idx]
    if name in BASELINES or name in STAT_MODELS:
        fn = BASELINES.get(name) or STAT_MODELS[name]
        hist = feat.loc[(feat._is_test == 0) & (feat[DATE_COL] < cutoff), ["_sid", DATE_COL, TARGET]]
        return per_series_forecast(fn, hist, tgt)
    tr = feat.loc[is_train_row & (feat[DATE_COL] < cutoff)]
    model = clone(ML[name]).fit(tr[FEATS], tr["_y"])
    p = model.predict(tgt[FEATS])
    return _clip(np.expm1(p) if LOG_TARGET else p)

va_idx = feat.index[(feat._is_test == 0) & (feat[DATE_COL] >= VALID_START)]
y_va = feat.loc[va_idx, TARGET].values
VALID_PRED, rows = {}, []
for name in list(BASELINES) + list(STAT_MODELS) + list(ML):
    t0 = time.time()
    VALID_PRED[name] = run_model(name, VALID_START, va_idx)
    rows.append({"model": name, METRIC: score(y_va, VALID_PRED[name]), "mae": score(y_va, VALID_PRED[name], "mae"),
                 "wape%": score(y_va, VALID_PRED[name], "wape"), "time_s": time.time() - t0})
    print(f"{name:24s} {METRIC} = {rows[-1][METRIC]:.4f}  ({rows[-1]['time_s']:.1f}s)")

leaderboard = pd.DataFrame(rows).sort_values(METRIC).reset_index(drop=True)
display(leaderboard)
plt.figure(figsize=(9, .4 * len(leaderboard) + 1))
colors = ["gray" if m in BASELINES else ("purple" if m in STAT_MODELS else "teal") for m in leaderboard.model]
plt.barh(leaderboard.model[::-1], leaderboard[METRIC][::-1], color=colors[::-1])
plt.title(f"Holdout {METRIC} (lower = better) — gray: baseline, purple: statistical, teal: ML"); plt.show()

## 9. Backtesting (একাধিক time fold) + Hyperparameter tuning
একটা holdout এ ভাগ্যক্রমে ভালো হতে পারে — কয়েকটা cutoff এ গড় score দেখি।

In [ ]:
def backtest(name):
    s = []
    for fs in FOLD_STARTS:
        idx = feat.index[(feat._is_test == 0) & (feat[DATE_COL] >= fs) & (feat[DATE_COL] < fs + H * STEP)]
        s.append(score(feat.loc[idx, TARGET].values, run_model(name, fs, idx)))
    return np.array(s)

bt_names = leaderboard.model.head(3).tolist() + ["Seasonal naive"]
bt = pd.DataFrame({n: backtest(n) for n in dict.fromkeys(bt_names)}, index=[f"fold {i} ({fs.date()})" for i, fs in enumerate(FOLD_STARTS)])
bt.loc["MEAN"] = bt.mean(); bt.loc["STD"] = bt.iloc[:-1].std()
display(bt.round(4))

In [ ]:
if optuna is not None and lgb is not None:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def objective(trial):
        ML["_trial"] = lgb.LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 1500, step=100),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 200, log=True),
            subsample=trial.suggest_float("subsample", 0.5, 1.0), subsample_freq=1,
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
            objective=trial.suggest_categorical("objective", ["regression", "tweedie"]),   # tweedie: অনেক zero থাকলে ভালো
            random_state=SEED, verbose=-1)
        return backtest("_trial").mean()
    study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_OPTUNA_TRIALS, timeout=900)
    ML.pop("_trial", None)
    print(f"Best backtest {METRIC}: {study.best_value:.4f}\n{study.best_params}")
    ML["LightGBM_tuned"] = lgb.LGBMRegressor(**study.best_params, subsample_freq=1, random_state=SEED, verbose=-1)
    VALID_PRED["LightGBM_tuned"] = run_model("LightGBM_tuned", VALID_START, va_idx)
    print(f"LightGBM_tuned holdout {METRIC} = {score(y_va, VALID_PRED['LightGBM_tuned']):.4f}")

## 10. Ensemble + Evaluation

In [ ]:
ranked = sorted(VALID_PRED, key=lambda n: score(y_va, VALID_PRED[n]))
CANDIDATES = {n: [n] for n in ranked[:3]}
CANDIDATES["AVG top-2"] = ranked[:2]
CANDIDATES["AVG top-3"] = ranked[:3]
ens_scores = {k: score(y_va, np.mean([VALID_PRED[m] for m in v], axis=0)) for k, v in CANDIDATES.items()}
display(pd.Series(ens_scores).sort_values().to_frame(f"holdout {METRIC}"))
BEST = min(ens_scores, key=ens_scores.get)
BEST_MEMBERS = CANDIDATES[BEST]
best_va = np.mean([VALID_PRED[m] for m in BEST_MEMBERS], axis=0)
print("🏆 Final choice:", BEST, "→", BEST_MEMBERS)

In [ ]:
ev = feat.loc[va_idx, ["_sid", DATE_COL, TARGET]].copy()
ev["pred"], ev["snaive"] = best_va, VALID_PRED["Seasonal naive"]
ev["step"] = ev.groupby("_sid").cumcount() + 1
ev["err"] = ev[TARGET] - ev["pred"]

# Forecast vs actual — কয়েকটা series
show = top_sids[:4]
fig, axes = plt.subplots(len(show), 1, figsize=(16, 3 * len(show)), sharex=True)
for ax, sid in zip(np.atleast_1d(axes), show):
    h_ = feat[(feat._sid == sid) & (feat._is_test == 0) & (feat[DATE_COL] < VALID_START)].tail(SEASON * 10)
    e_ = ev[ev._sid == sid]
    ax.plot(h_[DATE_COL], h_[TARGET], color="gray", lw=1, label="history")
    ax.plot(e_[DATE_COL], e_[TARGET], color="k", lw=2, label="actual")
    ax.plot(e_[DATE_COL], e_["pred"], color="tab:red", lw=2, label=f"{BEST}")
    ax.plot(e_[DATE_COL], e_["snaive"], "--", color="tab:blue", lw=1, label="seasonal naive")
    ax.set_title(sid); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
ev.groupby("step").apply(lambda g: score(g[TARGET], g["pred"])).plot(ax=axes[0], marker="o")
axes[0].set_title(f"{METRIC} by forecast step (দূরের দিন কি বেশি ভুল?)")
ev.groupby("_sid").apply(lambda g: score(g[TARGET], g["pred"], "wape")).sort_values().plot.barh(ax=axes[1])
axes[1].set_title("WAPE% by series")
sns.histplot(ev["err"], kde=True, ax=axes[2]); axes[2].set_title(f"Residuals (mean={ev['err'].mean():.2f} → bias)")
plt.tight_layout(); plt.show()

if sm is not None:
    res_t = ev.groupby(DATE_COL)["err"].sum()
    fig, axes = plt.subplots(1, 2, figsize=(16, 3.5))
    axes[0].plot(res_t.index, res_t.values); axes[0].axhline(0, c="r"); axes[0].set_title("Total residual over time")
    plot_acf(res_t, lags=min(20, len(res_t) - 2), ax=axes[1]); axes[1].set_title("Residual ACF (spike থাকলে pattern বাকি আছে)")
    plt.tight_layout(); plt.show()

## 11. Interpretation — feature importance + SHAP

In [ ]:
tree_name = next((n for n in ["LightGBM_tuned", "LightGBM", "XGBoost", "CatBoost", "ExtraTrees", "RandomForest"] if n in ML), None)
if tree_name:
    tr = feat.loc[is_train_row & (feat[DATE_COL] < VALID_START)]
    m = clone(ML[tree_name]).fit(tr[FEATS], tr["_y"])
    est = m[-1] if hasattr(m, "steps") else m                               # pipeline হলে শেষ step
    imp = pd.Series(est.feature_importances_, FEATS).sort_values()
    imp.tail(25).plot.barh(figsize=(8, 7), title=f"{tree_name} feature importance"); plt.show()
    if shap is not None:
        Xs = feat.loc[va_idx, FEATS].sample(min(1000, len(va_idx)), random_state=SEED)
        if hasattr(m, "steps"):
            Xs = pd.DataFrame(m[:-1].transform(Xs), columns=FEATS)
        sv = shap.TreeExplainer(est)(Xs)
        shap.plots.beeswarm(sv, max_display=20, show=False); plt.title("SHAP summary"); plt.show()

## 12. Final forecast (পুরো train দিয়ে refit) → Submission

In [ ]:
if test is not None:
    te_idx = feat.index[feat._is_test == 1]
    cutoff = TRAIN_END + STEP
    test_pred = np.mean([run_model(m, cutoff, te_idx) for m in BEST_MEMBERS], axis=0)
    feat.loc[te_idx, "forecast"] = test_pred

    fig, axes = plt.subplots(len(show), 1, figsize=(16, 3 * len(show)), sharex=True)
    for ax, sid in zip(np.atleast_1d(axes), show):
        h_ = feat[(feat._sid == sid) & (feat._is_test == 0)].tail(SEASON * 12)
        f_ = feat[(feat._sid == sid) & (feat._is_test == 1)]
        ax.plot(h_[DATE_COL], h_[TARGET], color="gray", lw=1, label="history")
        ax.plot(f_[DATE_COL], f_["forecast"], color="tab:red", lw=2, label="forecast")
        ax.set_title(sid); ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout(); plt.show()

    out = feat.loc[te_idx, [c for c in [ROW_ID_COL, "_sid", DATE_COL] if c] + ["forecast"]]
    if ROW_ID_COL and ROW_ID_COL in test:
        out[ROW_ID_COL] = out[ROW_ID_COL].astype(test[ROW_ID_COL].dtype)   # concat এ int → float হয়েছিল
        sub = test[[ROW_ID_COL]].merge(out[[ROW_ID_COL, "forecast"]], on=ROW_ID_COL, how="left")
        sub.columns = sample_sub.columns[:2] if sample_sub is not None else [ROW_ID_COL, TARGET]
    else:
        sub = out.rename(columns={"forecast": TARGET})
    assert sub.iloc[:, -1].notna().all(), "forecast এ NaN আছে!"
    if sample_sub is not None:
        assert len(sub) == len(sample_sub), "row সংখ্যা sample_submission এর সাথে মিলছে না!"
    sub.to_csv(OUTPUT_DIR / "submission.csv", index=False)
    print("Saved →", OUTPUT_DIR / "submission.csv", sub.shape)
    display(sub.head()); print(sub.iloc[:, -1].describe())

## ☁️ Colab: Submission download + Drive এ backup

In [ ]:
SAVE_TO_DRIVE = False                                  # True → outputs folder Google Drive এ copy হবে
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/datathon/outputs/timeseries"

if IN_COLAB:
    from google.colab import files, drive
    sub_path = OUTPUT_DIR / "submission.csv"
    if sub_path.exists():
        files.download(str(sub_path))                  # browser এ download হবে
    if SAVE_TO_DRIVE:
        drive.mount("/content/drive")
        shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print("Saved to Drive →", DRIVE_OUTPUT_DIR)
else:
    print("Local run — files আছে:", OUTPUT_DIR.resolve())

## 13. অন্য রকম Time-series task হলে?

| Task | কি করবো |
|---|---|
| **একটাই series** | `ID_COLS = []` — বাকি সব একই |
| **Monthly / weekly data** | `FREQ="MS"`/`"W"`, `SEASON=12`/`52`; data কম হলে ETS/SARIMA প্রায়ই ML এর চেয়ে ভালো |
| **Hourly data** | `FREQ="h"`, `SEASON=24` (weekly pattern ও থাকলে lag 168 যোগ করো) |
| **H খুব বড় (lag ≥ H রাখলে তথ্য পুরনো)** | **Recursive:** lag_1 দিয়ে ১ step predict → সেটা history তে বসিয়ে পরের step; অথবা প্রতিটা horizon এর জন্য আলাদা model |
| **অনেক zero (intermittent demand)** | LightGBM `objective="tweedie"`, Croston method, metric WAPE/RMSSE |
| **Time-series classification** (sensor/ECG/activity) | window কেটে feature (mean, std, min, max, fft, slope) → tabular template; বা `tsfresh` / 1D-CNN |
| **Anomaly detection** | STL residual / rolling z-score (Section 2.3) / IsolationForest |
| **Deep learning / AutoML** | `statsforecast` (দ্রুত AutoARIMA/AutoETS), `neuralforecast` (N-BEATS, NHITS, TFT), `darts`, `sktime` |
| **Hierarchical** (store → region → total) | প্রতিটা level এ forecast করে reconcile (bottom-up / MinT) |

## ✅ Score বাড়ানোর checklist
1. **Seasonal naive কে হারাতে পারছো তো?** না পারলে feature/validation এ সমস্যা
2. Lag + rolling + calendar + holiday + promo — feature engineering এ সময় দাও
3. `LOG_TARGET=True` / tweedie objective (sales data)
4. Backtest এর **গড়** score দিয়ে model বাছো, একটা holdout দিয়ে না
5. ML (LightGBM) + ETS/seasonal average এর ensemble প্রায়ই সবচেয়ে stable
6. Final model এ **validation period সহ পুরো data** দিয়ে refit (Section 12 তে করা আছে)